# skforecast-ai — Demo Phase 3

This notebook demonstrates the **Phase 3: Recommendation Engine**.

Given a `DataProfile` and a forecast horizon, `recommend_plan` produces a fully
deterministic `ForecastPlan` using rule-based logic derived from the skforecast
skills. No LLM is involved.

## Setup

In [1]:
import numpy as np
import pandas as pd

import skforecast_ai
from skforecast_ai.profiling import create_data_profile
from skforecast_ai.recommendation import recommend_plan

print(f"skforecast-ai version: {skforecast_ai.__version__}")

skforecast-ai version: 0.1.0


## 1. Single series — daily (default recommendation)

In [2]:
df_daily = pd.DataFrame(
    {"y": np.random.default_rng(42).normal(100, 10, 365)},
    index=pd.date_range("2023-01-01", periods=365, freq="D"),
)

profile = create_data_profile(data=df_daily, target="y")
profile

DataProfile(n_observations=365, n_series=1, index_type='datetime', frequency='D', target='y', date_column=None, series_id_column=None, exog_columns=[], categorical_exog=[], missing_values={}, inferred_seasonalities=[7, 365], warnings=[])

In [3]:
plan = recommend_plan(profile=profile, horizon=30)
plan

ForecastPlan(task_type='single_series', forecaster='ForecasterRecursive', estimator='Ridge', horizon=30, frequency='D', lags=[1, 2, 3, 4, 5, 6, 7], metric='mean_absolute_error', backtesting_strategy='TimeSeriesFold', interval_method='bootstrapping', use_exog=False, data_requirements=[], warnings=[], rationale='A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping.')

In [4]:
print(f"Task type:    {plan.task_type}")
print(f"Forecaster:   {plan.forecaster}")
print(f"Estimator:    {plan.estimator}")
print(f"Lags:         {plan.lags}")
print(f"Metric:       {plan.metric}")
print(f"Backtesting:  {plan.backtesting_strategy}")
print(f"Intervals:    {plan.interval_method}")
print(f"Use exog:     {plan.use_exog}")
print(f"\nRationale: {plan.rationale}")

Task type:    single_series
Forecaster:   ForecasterRecursive
Estimator:    Ridge
Lags:         [1, 2, 3, 4, 5, 6, 7]
Metric:       mean_absolute_error
Backtesting:  TimeSeriesFold
Intervals:    bootstrapping
Use exog:     False

Rationale: A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping.


## 2. Single series — hourly with exogenous variables

In [5]:
rng = np.random.default_rng(42)
df_hourly = pd.DataFrame(
    {
        "sales": rng.normal(500, 50, 720),
        "temperature": np.tile(np.linspace(5, 35, 24), 30),
        "promo_budget": rng.uniform(100, 1000, 720),
        "holiday": ["no"] * 700 + ["yes"] * 20,
    },
    index=pd.date_range("2023-01-01", periods=720, freq="h"),
)

profile = create_data_profile(data=df_hourly, target="sales")
plan = recommend_plan(profile=profile, horizon=24)

print(f"Forecaster:  {plan.forecaster}")
print(f"Estimator:   {plan.estimator}")
print(f"Lags:        {plan.lags}")
print(f"Use exog:    {plan.use_exog}")
print(f"Intervals:   {plan.interval_method}")
print(f"\nData requirements: {plan.data_requirements}")

Forecaster:  ForecasterRecursive
Estimator:   LGBMRegressor
Lags:        [1, 2, 3, 4, 5, 6, 7, 24, 168]
Use exog:    True
Intervals:   bootstrapping

Data requirements: ['Encode categorical exogenous variables or use an estimator with native categorical support (e.g. LightGBM, CatBoost).']


## 3. Multi-series — long format

In [6]:
dates = pd.date_range("2023-01-01", periods=100, freq="D")
df_multi = pd.DataFrame(
    {
        "date": np.tile(dates, 3),
        "series_id": np.repeat(["store_A", "store_B", "store_C"], 100),
        "sales": rng.normal(200, 30, 300),
        "promo": rng.choice([0, 1], 300),
    }
)

profile = create_data_profile(
    data=df_multi,
    target="sales",
    date_column="date",
    series_id_column="series_id",
)
plan = recommend_plan(profile=profile, horizon=14)

print(f"Task type:   {plan.task_type}")
print(f"Forecaster:  {plan.forecaster}")
print(f"Intervals:   {plan.interval_method}")
print(f"Use exog:    {plan.use_exog}")
print(f"\nRationale: {plan.rationale}")

Task type:   multi_series
Forecaster:  ForecasterRecursiveMultiSeries
Intervals:   conformal
Use exog:    True

Rationale: The dataset contains 3 series, so a multi-series forecaster (ForecasterRecursiveMultiSeries) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via conformal. Exogenous variables detected: ['promo'].


## 4. User preferences — foundation model

In [7]:
profile = create_data_profile(data=df_daily, target="y")
plan = recommend_plan(profile=profile, horizon=30, prefer_foundation=True)

print(f"Task type:   {plan.task_type}")
print(f"Forecaster:  {plan.forecaster}")
print(f"Estimator:   {plan.estimator}")
print(f"Lags:        {plan.lags}")
print(f"Intervals:   {plan.interval_method}")
print(f"\nRationale: {plan.rationale}")

Task type:   foundation
Forecaster:  ForecasterFoundation
Estimator:   None
Lags:        None
Intervals:   None

Rationale: A foundation model (ForecasterFoundation) was selected per user preference. Metric: mean_absolute_error.


## 5. User preferences — statistical model

In [8]:
plan = recommend_plan(profile=profile, horizon=30, prefer_statistical=True)

print(f"Task type:   {plan.task_type}")
print(f"Forecaster:  {plan.forecaster}")
print(f"Estimator:   {plan.estimator}")
print(f"Lags:        {plan.lags}")
print(f"\nRationale: {plan.rationale}")

Task type:   statistical
Forecaster:  ForecasterStats
Estimator:   None
Lags:        None

Rationale: A statistical model (ForecasterStats) was selected per user preference. Metric: mean_absolute_error.


## 6. Edge cases — warnings and data requirements

In [9]:
# Horizon larger than data
profile = create_data_profile(data=df_daily, target="y")
plan = recommend_plan(profile=profile, horizon=500)

print("Warnings:")
for w in plan.warnings:
    print(f"  - {w}")

Warnings:
  - Horizon (500) exceeds the number of observations (365). Results may be unreliable.


In [10]:
# Short series with missing values and no datetime
df_edge = pd.DataFrame({"value": [1.0, 2.0, np.nan, 4.0] * 10})
profile = create_data_profile(data=df_edge, target="value")
plan = recommend_plan(profile=profile, horizon=5)

print(f"Estimator:   {plan.estimator}")
print(f"Intervals:   {plan.interval_method}")
print(f"\nData requirements:")
for req in plan.data_requirements:
    print(f"  - {req}")
print(f"\nWarnings: {plan.warnings}")

Estimator:   Ridge
Intervals:   None

Data requirements:
  - Impute missing values before training.
  - Provide a DatetimeIndex or date column for time-based features.

Warnings: []


## 7. Determinism verification

In [11]:
profile = create_data_profile(data=df_daily, target="y")

plan_1 = recommend_plan(profile=profile, horizon=30)
plan_2 = recommend_plan(profile=profile, horizon=30)

assert plan_1 == plan_2
print("Determinism OK: identical inputs produce identical outputs.")

Determinism OK: identical inputs produce identical outputs.


## 8. JSON serialization

In [12]:
json_str = plan_1.model_dump_json(indent=2)
print(json_str)

{
  "task_type": "single_series",
  "forecaster": "ForecasterRecursive",
  "estimator": "Ridge",
  "horizon": 30,
  "frequency": "D",
  "lags": [
    1,
    2,
    3,
    4,
    5,
    6,
    7
  ],
  "metric": "mean_absolute_error",
  "backtesting_strategy": "TimeSeriesFold",
  "interval_method": "bootstrapping",
  "use_exog": false,
  "data_requirements": [],
  "warnings": [],
  "rationale": "A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping."
}


In [13]:
from skforecast_ai.schemas import ForecastPlan

restored = ForecastPlan.model_validate_json(json_str)
assert restored == plan_1
print("JSON roundtrip OK")

JSON roundtrip OK
